# Лабораторная работа №5. Воспроизводимый эксперимент с YOLO

**Цель:** пройти полный цикл детекции — контракт данных → baseline → контролируемое изменение → оценка → анализ ошибок — на собственном датасете.

После работы обучающийся сможет:

1. проверить YOLO-разметку и зафиксировать train/validation/test split;
2. запустить предобученную YOLO Nano/Small через единый конфиг;
3. интерпретировать precision, recall, F1, mAP@0.5 и mAP@0.5:0.95;
4. сформулировать гипотезу **до** запуска и изменить ровно один фактор;
5. сохранить артефакты так, чтобы другой студент воспроизвёл результат.

**Бюджет:** smoke — `coco8.yaml`, CPU, 1 эпоха, 320 px; full — согласованный датасет, один GPU, обычно 20–50 эпох, модель не крупнее Small.

**Критерий завершения:** baseline и одна контролируемая итерация на одном validation split, полный журнал конфигураций, количественный и визуальный анализ ошибок.

## 1. Методика

YOLO — одностадийный детектор: за один проход он предсказывает координаты, objectness и классы. Скорость не отменяет корректного протокола. Рабочий порог confidence подбирается по validation; test используется один раз после выбора конфигурации.

До обучения запишите:

- предметную область и цену FP/FN;
- целевую метрику и почему она соответствует сценарию;
- проверяемую гипотезу с ожидаемым направлением эффекта;
- один изменяемый фактор: например, разрешение, сила аугментации или размер модели.

Нельзя одновременно менять модель, разрешение, эпохи и состав данных: такой прирост невозможно интерпретировать.

## 2. Окружение и режимы

- `CV_SMOKE=1` — публичный tiny dataset `coco8.yaml`, CPU, одна эпоха;
- `CV_SMOKE=0` и `YOLO_DATASET=/путь/data.yaml` — полный режим;
- `CV_RUN_TRAINING=1` — разрешить обучение после заполнения TODO;
- `CV_PUBLIC_CHECKS=1` — выполнить открытые проверки.

Для полного режима зафиксируйте версию датасета и лицензию. Скачивание «последней версии» без идентификатора недопустимо.

In [ ]:
import json
import os
import random
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import torch
import yaml
from ultralytics import YOLO

SEED = 42
SMOKE_MODE = os.getenv("CV_SMOKE", "1") == "1"
RUN_TRAINING = os.getenv("CV_RUN_TRAINING", "0") == "1"
RUN_PUBLIC_CHECKS = os.getenv("CV_PUBLIC_CHECKS", "0") == "1"
OUTPUT_ROOT = Path("outputs/yolo")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_global_seed()
print({"smoke": SMOKE_MODE, "training_enabled": RUN_TRAINING})

## 3. Контракт датасета

Полный `data.yaml` обязан содержать `path`, `train`, `val`, `test`, `names`. Разбиение создаётся один раз с `SEED=42` и больше не меняется между итерациями. Для кадров одного видео делите по видео/сценам, а не случайно по кадрам, иначе возникает утечка.

Сохраните `dataset-card.md`: источник, версия, лицензия, число изображений и объектов по классам/сплитам, правила разбиения, известные смещения и обработка персональных данных. Сохраните также `split_manifest.json` со списками файлов или их хешами.

In [ ]:
SMOKE_DATASET = "coco8.yaml"
FULL_DATASET = Path(os.getenv("YOLO_DATASET", "data/detection/data.yaml"))
DATA_CONFIG = SMOKE_DATASET if SMOKE_MODE else str(FULL_DATASET)

REQUIRED_DATA_KEYS = {"path", "train", "val", "test", "names"}

def validate_dataset_contract(data_config: str) -> dict:
    """Проверьте структуру локального data.yaml; встроенный coco8 проверяет Ultralytics."""
    if data_config == SMOKE_DATASET:
        return {"kind": "builtin-smoke", "name": data_config}

    config_path = Path(data_config)
    assert config_path.is_file(), f"Не найден data.yaml: {config_path}"
    payload = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    missing = REQUIRED_DATA_KEYS - set(payload)
    assert not missing, f"В data.yaml отсутствуют ключи: {sorted(missing)}"
    assert payload["train"] != payload["val"], "train и val не должны совпадать"
    assert payload["test"] not in (payload["train"], payload["val"]), "test должен быть отдельным"
    assert payload["names"], "Список классов пуст"
    return payload

dataset_contract = validate_dataset_contract(DATA_CONFIG)
print(dataset_contract)

## 4. Единый конфиг запуска

Все параметры, влияющие на результат, находятся в `RunConfig`. Baseline и вторая итерация должны отличаться ровно одним полем из списка `CONTROLLED_FIELDS`. Поля `name` и `project` служебные и не считаются исследуемым фактором.

In [ ]:
@dataclass(frozen=True)
class RunConfig:
    name: str
    data: str
    model: str
    epochs: int
    imgsz: int
    batch: int
    device: str
    seed: int = SEED
    workers: int = 0
    patience: int = 10
    optimizer: str = "auto"
    lr0: float = 0.01
    degrees: float = 0.0
    mosaic: float = 1.0

BASELINE = RunConfig(
    name="baseline_smoke" if SMOKE_MODE else "baseline",
    data=DATA_CONFIG,
    model="yolo11n.pt",
    epochs=1 if SMOKE_MODE else 30,
    imgsz=320 if SMOKE_MODE else 640,
    batch=4 if SMOKE_MODE else 16,
    device="cpu" if SMOKE_MODE else ("0" if torch.cuda.is_available() else "cpu"),
)

# TODO 1: выберите ОДИН фактор и сформулируйте гипотезу до запуска.
RESEARCH_QUESTION = ""
HYPOTHESIS = ""
CHANGED_FACTOR = ""  # например: "imgsz", "model", "mosaic"

# TODO 2: создайте SECOND_ITERATION через replace(BASELINE, ...),
# меняя name и ровно одно исследуемое поле.
SECOND_ITERATION = None

CONTROLLED_FIELDS = (
    "data", "model", "epochs", "imgsz", "batch", "device", "seed",
    "workers", "patience", "optimizer", "lr0", "degrees", "mosaic",
)

def changed_fields(first: RunConfig, second: RunConfig) -> list[str]:
    return [field for field in CONTROLLED_FIELDS
            if getattr(first, field) != getattr(second, field)]

print(asdict(BASELINE))

## 5. Starter scaffold: обучение и оценка

Реализуйте `train_and_evaluate` через публичный API `ultralytics.YOLO`.

Контракт функции:

1. создать `YOLO(config.model)`;
2. вызвать `model.train(...)`, передав все существенные поля конфига, `project=OUTPUT_ROOT`, `name=config.name`, `exist_ok=False`, `deterministic=True`;
3. вызвать `model.val(data=config.data, split="val", ...)` на том же разрешении и устройстве;
4. извлечь метрики функцией `extract_detection_metrics`;
5. сохранить `config.json` и `metrics.json` в каталоге запуска;
6. вернуть плоский словарь для итоговой таблицы.

Не копируйте готовую команду из чужого отчёта: заполните именованные аргументы из `RunConfig` и объясните каждый изменяемый параметр.

In [ ]:
def extract_detection_metrics(metrics) -> dict:
    """Нормализует ключевые метрики объекта Results, возвращаемого model.val()."""
    return {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
    }

def save_run_record(config: RunConfig, metrics: dict) -> Path:
    run_dir = OUTPUT_ROOT / config.name
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(
        json.dumps(asdict(config), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (run_dir / "metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return run_dir

def train_and_evaluate(config: RunConfig) -> dict:
    """Обучите и оцените один запуск строго по RunConfig."""
    set_global_seed(config.seed)

    # TODO 3: создайте model = YOLO(...)
    # TODO 4: вызовите model.train с параметрами config.
    # TODO 5: вызовите model.val на validation split.
    # TODO 6: извлеките метрики, добавьте name/model/imgsz и сохраните артефакты.
    # Важно: test здесь не используется.
    raise NotImplementedError("Реализуйте YOLO train/validation pipeline")

run_results = []
if RUN_TRAINING:
    assert SECOND_ITERATION is not None, "Сначала задайте вторую итерацию"
    for config in (BASELINE, SECOND_ITERATION):
        run_results.append(train_and_evaluate(config))
else:
    print("Обучение выключено. Завершите TODO и задайте CV_RUN_TRAINING=1.")

## 6. Анализ ошибок и финальный test

По validation:

- постройте PR-кривую и confusion matrix;
- покажите минимум по три FP и FN;
- разделите ошибки на категории: локализация, классификация, пропуск мелкого/перекрытого объекта, ошибка разметки;
- подберите рабочий confidence threshold по цене FP/FN.

Только после выбора лучшей конфигурации по validation выполните **один** вызов `model.val(split="test")`. Если у датасета нет независимого test, честно укажите это ограничение и не называйте validation-метрику тестовой.

In [ ]:
# TODO 7: соберите таблицу baseline/iteration-2 с одинаковыми колонками.
# TODO 8: выберите лучшую конфигурацию ТОЛЬКО по validation.
# TODO 9: один раз оцените выбранный checkpoint на test.
# TODO 10: сохраните summary.csv и error_analysis.md в OUTPUT_ROOT.

# Пример ожидаемых колонок:
SUMMARY_COLUMNS = [
    "name", "model", "imgsz", "precision", "recall", "map50", "map50_95"
]

## 7. Открытые проверки

Проверки не оценивают mAP и не раскрывают решения. Они ловят несогласованный конфиг, отсутствие независимых сплитов и изменение нескольких факторов одновременно.

In [ ]:
def run_public_checks() -> None:
    assert SEED == 42
    assert BASELINE.seed == SEED
    assert BASELINE.epochs == (1 if SMOKE_MODE else 30)
    if SMOKE_MODE:
        assert BASELINE.data == "coco8.yaml"
        assert BASELINE.device == "cpu"
        assert BASELINE.imgsz <= 320

    assert RESEARCH_QUESTION.strip(), "Сформулируйте исследовательский вопрос"
    assert HYPOTHESIS.strip(), "Сформулируйте гипотезу до запуска"
    assert SECOND_ITERATION is not None, "Создайте SECOND_ITERATION"
    differences = changed_fields(BASELINE, SECOND_ITERATION)
    assert differences == [CHANGED_FACTOR], (
        f"Ожидалось изменение только {CHANGED_FACTOR!r}, получено: {differences}"
    )
    assert SECOND_ITERATION.name != BASELINE.name
    validate_dataset_contract(BASELINE.data)
    print("Public checks: OK")

if RUN_PUBLIC_CHECKS:
    run_public_checks()
else:
    print("Public checks определены; включение: CV_PUBLIC_CHECKS=1.")

## 8. Сдаваемые артефакты и критерии

Обязательная структура:

```text
outputs/yolo/
├── split_manifest.json
├── summary.csv
├── error_analysis.md
├── baseline/
│   ├── config.json
│   ├── metrics.json
│   └── ... артефакты Ultralytics
└── iteration_2/
    ├── config.json
    ├── metrics.json
    └── ... артефакты Ultralytics
```

К отчёту приложите dataset card, гипотезу до запуска, одинаковые метрики двух итераций, PR-кривые/confusion matrix, примеры FP/FN, выбранный threshold, стоимость инференса и ограничения.

Работа не засчитывается, если:

- split менялся между итерациями или кадры одной сцены попали в разные split;
- изменено больше одного фактора без отдельной абляции;
- лучшая конфигурация или threshold выбраны по test;
- отсутствуют конфиги и артефакты, позволяющие повторить запуск;
- вывод основан только на нескольких красивых изображениях.

### Контрольные вопросы

1. Чем confidence threshold отличается от IoU threshold при расчёте AP?
2. Почему mAP@0.5:0.95 сильнее штрафует неточную локализацию, чем mAP@0.5?
3. Как scene-level split предотвращает утечку для кадров видео?
4. Почему изменение двух факторов не позволяет приписать прирост одному из них?
5. В каком прикладном сценарии recall важнее precision и наоборот?